# 01. Connect to the Exchange and Inventory Available Data

This notebook enumerates every study, patient, consent scope, and observation type visible to the logged-in account, then writes an aggregate snapshot to `outputs/inventory_snapshot.csv`. The snapshot contains counts only and is safe to commit.

Run it on the JupyterHealth Hub, where `JHE_URL` and `JHE_TOKEN` are already set.

In [ ]:
import pandas as pd
from jupyterhealth_client import JupyterHealthClient, Code

client = JupyterHealthClient()
client.get_user()

## Studies

In [ ]:
studies = pd.DataFrame([
    {"study_id": s["id"], "study_name": s["name"], "organization": s["organization"]["name"]}
    for s in client.list_studies()
])
studies

## Patients and consented scopes

One row per patient, study, and consented measurement type. Consent is granted per study and per scope, so this table shows exactly which data each study is permitted to see.

In [ ]:
rows = []
for s in studies.itertuples():
    for p in client.list_patients(study_id=s.study_id):
        consents = client.get_patient_consents(p["id"])
        for study in consents["studies"]:
            if study["id"] != s.study_id:
                continue
            for scope in study["scopeConsents"]:
                rows.append({
                    "study_id": s.study_id,
                    "patient_id": p["id"],
                    "scope": scope["code"]["codingCode"],
                    "consented": scope["consented"],
                })

consents = pd.DataFrame(rows)
consents.groupby(["study_id", "scope", "consented"]).patient_id.nunique().unstack(fill_value=0)

## Observation counts by study and type

`limit=None` retrieves every record, which can be slow for large studies. Reduce it if a study is very large.

Observation types record time differently: point-in-time measures (blood pressure, glucose) carry a single timestamp, while interval measures (step count, sleep, physical activity) carry a start and end. The helper below detects whichever column is present, and any study and type combination that the Exchange cannot return is listed separately instead of halting the loop.

In [ ]:
def find_time_col(df):
    """Return the timestamp column for this observation type.

    Point-in-time measures (blood pressure, glucose) use effective_time_frame_date_time;
    interval measures (steps, sleep, physical activity) use an interval start column.
    """
    preferred = [
        "effective_time_frame_date_time",
        "effective_time_frame_time_interval_start_date_time",
        "effective_time_frame_time_interval_end_date_time",
    ]
    for c in preferred:
        if c in df.columns:
            return c
    return next((c for c in df.columns if "date_time" in c), None)


summary, skipped = [], []
for s in studies.itertuples():
    for code in Code:
        try:
            obs = client.list_observations_df(study_id=s.study_id, code=code, limit=None)
        except Exception as e:
            skipped.append({"study_id": s.study_id, "code": code.value, "error": type(e).__name__})
            continue
        if obs is None or len(obs) == 0:
            continue

        tcol = find_time_col(obs)
        t = pd.to_datetime(obs[tcol], errors="coerce", utc=True) if tcol else pd.Series(dtype="datetime64[ns, UTC]")

        summary.append({
            "study_id": s.study_id,
            "study_name": s.study_name,
            "code": code.value,
            "n_patients": obs["subject_reference"].nunique() if "subject_reference" in obs.columns else pd.NA,
            "n_observations": len(obs),
            "time_column": tcol,
            "first": t.min().date() if t.notna().any() else pd.NA,
            "last": t.max().date() if t.notna().any() else pd.NA,
        })

summary = pd.DataFrame(summary).sort_values(["study_id", "code"]).reset_index(drop=True)
display(summary)

if skipped:
    print("Skipped (no data or no schema for this study):")
    display(pd.DataFrame(skipped))

In [ ]:
import os

os.makedirs("../outputs", exist_ok=True)
summary.to_csv("../outputs/inventory_snapshot.csv", index=False)

try:
    print(summary.to_markdown(index=False))  # paste into docs/02_data-inventory.md
except ImportError:
    print("Install `tabulate` for markdown output; CSV was saved.")

## What is in each study

The study name is only a label. The cells below describe each study through its stored metadata, the fields each observation type records, the distribution of recorded values, and how consistently each patient reports. The examples use blood glucose in study 30006, the richest demo study; swap in any study and `Code` member.

In [ ]:
# 1. Full study metadata: every field the Exchange stores, not just the name
raw_studies = list(client.list_studies())
print(raw_studies[0].keys())          # see which fields exist (description, icon, organization, ...)
pd.json_normalize(raw_studies)

In [ ]:
# 2. Column schema per observation type: what each measurement actually records
def schema(study_id, code):
    df = client.list_observations_df(study_id=study_id, code=code, limit=5)
    return [c for c in df.columns if not c.startswith(("resource_", "meta_"))]

schema(30006, Code.BLOOD_GLUCOSE)     # swap in any Code member

In [ ]:
# 3. Value distributions: units, plausible ranges, and whether the data looks synthetic
glucose = client.list_observations_df(study_id=30006, code=Code.BLOOD_GLUCOSE, limit=None)
value_cols = [c for c in glucose.columns if c.endswith(("_value", "_unit"))]
glucose[value_cols].describe(include="all")

In [ ]:
# 4. Per-patient coverage: how many days each patient reports, the basis for any dropout model
glucose = client.list_observations_df(study_id=30006, code=Code.BLOOD_GLUCOSE, limit=None)
tcol = find_time_col(glucose)
glucose["day"] = pd.to_datetime(glucose[tcol], utc=True).dt.date

coverage = (glucose.groupby("subject_reference")
              .agg(n_obs=("day", "size"), n_days=("day", "nunique"),
                   first=("day", "min"), last=("day", "max")))
coverage.describe()

## What else the Exchange exposes

These answer the open questions in `docs/02_data-inventory.md`: which FHIR resource types exist beyond Observation, and which data sources are registered.

In [ ]:
client.list_fhir_resource_types()

In [ ]:
pd.DataFrame(list(client.list_data_sources()))